In [1]:
import sys
from pathlib import Path
raiz = Path.cwd()
while not (raiz / "src").exists() and raiz != raiz.parent:
    raiz = raiz.parent
sys.path.insert(0, str(raiz))
print("raíz del proyecto:", raiz)

import numpy as np
import pandas as pd
from src.cuantizacion import (ajustar_cuantizador, transformar, reconstruir,
                              calcular_error)
DIR_TABLAS = raiz / "resultados" / "actividad3" / "tablas"
DIR_TABLAS.mkdir(parents=True, exist_ok=True)

raíz del proyecto: C:\Users\alexi\Desktop\LIDI\Actividad_2


In [2]:
x = [10, 12, 15, 21, 24, 31, 35, 40]
m = ajustar_cuantizador(x, "uniforme", 4, columna="demo")
codigos = transformar(x, m)
rec = reconstruir(codigos, m)           # punto medio por defecto
print("reconstrucción:", rec)           # [13.75 13.75 13.75 21.25 21.25 28.75 36.25 36.25]
print(calcular_error(x, rec))           # mae 2.125, rmse ~2.424, error_max 3.75

reconstrucción: [13.75 13.75 13.75 21.25 21.25 28.75 36.25 36.25]
{'mae': 2.125, 'rmse': 2.4238399287081647, 'error_max': 3.75, 'n_validos': 8}


In [3]:
demo = pd.DataFrame({
    "original": x,
    "codigo": codigos,
    "reconstruido": rec,
    "error": np.array(rec) - np.array(x, dtype=float),
})
demo.to_csv(DIR_TABLAS / "demo_uniforme.csv", index=False)
demo

,original,codigo,reconstruido,error
0,10,0,13.75,3.75
1,12,0,13.75,1.75
2,15,0,13.75,-1.25
3,21,1,21.25,0.25
4,24,1,21.25,-2.75
5,31,2,28.75,-2.25
6,35,3,36.25,1.25
7,40,3,36.25,-3.75


In [4]:
rng = np.random.default_rng(0)
datos = rng.normal(loc=40, scale=15, size=100_000)

filas = []
for k in [5, 10, 25, 50]:
    mk = ajustar_cuantizador(datos, "uniforme", k)
    e = calcular_error(datos, reconstruir(transformar(datos, mk), mk))
    filas.append({"bins": k, "bins_reales": mk.n_bins_real, **e})
barrido = pd.DataFrame(filas)
barrido.to_csv(DIR_TABLAS / "barrido_bins_uniforme.csv", index=False)
barrido

,bins,bins_reales,mae,rmse,error_max,n_validos
0,5,5,6.928871,7.991346,13.839112,100000
1,10,10,3.445737,3.981514,6.919556,100000
2,25,25,1.385105,1.599220,2.767822,100000
3,50,50,0.692251,0.799367,1.383911,100000


In [5]:
y = [10, np.nan, 40, -5, 100]
cod_y, diag = transformar(y, m, devolver_diagnostico=True)
print(cod_y, diag["fuera_rango"])
print(reconstruir(cod_y, m))            # el NaN vuelve a NaN

[ 0 -1  3  0  3] [False False False  True  True]
[13.75   nan 36.25 13.75 36.25]


In [6]:
git add .
git commit -m "Reconstruccion con punto medio, calcular_error y demo uniforme"

SyntaxError: invalid syntax (3067279082.py, line 1)